# 01 - French Motor Vehicle Data Load and Basic Checks

In [1]:
# Import LIbraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
import statsmodels.api as sm
import statsmodels.formula.api as smf
from scipy.stats import chi2

# Display all columns
pd.set_option('display.max_columns', None)

In [2]:
# Import Data
fmv_freq_data = fetch_openml(data_id=41214, as_frame=True)
fmv_sev_data = fetch_openml(data_id=41215, as_frame=True)

fmv_freq_df = fmv_freq_data.frame
fmv_sev_df = fmv_sev_data.frame

fmv_sev_df.head()

,IDpol,ClaimAmount
0,1552,995.20
1,1010996,1128.12
2,4024277,1851.11
3,4007252,1204.00
4,4046424,1204.00


### Variable Definitions

In [3]:
# Exposure: Portion of a year the policy was active (Exposure becomes an offset term in the frequency model)
# ClaimNb: Number of claims reported during the exposure period (Target variable for frequency modeling)
# ClaimAmount: Total amount paid for claims on that policy (Target variable for severity modeling)
# DriverAge: Age of the driver (Risk typically decreases until middle age then increases)
# BonusMalus: French no-claims discount system (Indicates driver risk history)
# VehAge: Age of the insured vehicle (Older vehicles -> fewer claims, higher severity)
# VehPower: Engine power category (Higher power vehicles → higher claim frequency)
# Area: Type of area where the vehicle is primarily used (Urban areas → higher claim frequency)
# Density: Population density of policyholder location
# Region: Administrative region of France


In [4]:
# Severity data check

fmv_sev_df.shape

############################## Takeaway ############################## 
# This dataset is clean. 
# There are no missing values


(26639, 2)

In [5]:
# Merge frequency and severity dataframes
fmv_df = fmv_freq_df.merge(fmv_sev_df, how="left", on="IDpol")
fmv_df.head()

,IDpol,ClaimNb,Exposure,Area,VehPower,VehAge,DrivAge,BonusMalus,VehBrand,VehGas,Density,Region,ClaimAmount
0,1.0,1,0.10,D,5,0,55,50,B12,'Regular',1217,R82,NaN
1,3.0,1,0.77,D,5,0,55,50,B12,'Regular',1217,R82,NaN
2,5.0,1,0.75,B,6,2,52,50,B12,'Diesel',54,R22,NaN
3,10.0,1,0.09,B,7,0,46,50,B12,'Diesel',76,R72,NaN
4,11.0,1,0.84,B,7,0,46,50,B12,'Diesel',76,R72,NaN


In [6]:
# One more data check
fmv_df.describe()

,IDpol,ClaimNb,Exposure,VehPower,VehAge,DrivAge,BonusMalus,Density,ClaimAmount
count,6.795130e+05,679513.000000,679513.000000,679513.000000,679513.000000,679513.000000,679513.000000,679513.000000,2.644400e+04
mean,2.621743e+06,0.058554,0.529078,6.454486,7.043808,45.497617,59.784621,1794.052255,2.265513e+03
std,1.641757e+06,0.279196,0.364378,2.050990,5.664855,14.140106,15.666953,3960.829098,2.937103e+04
min,1.000000e+00,0.000000,0.002732,4.000000,0.000000,18.000000,50.000000,1.000000,1.000000e+00
25%,1.157893e+06,0.000000,0.180000,5.000000,2.000000,34.000000,50.000000,92.000000,6.859925e+02
50%,2.272111e+06,0.000000,0.490000,6.000000,6.000000,44.000000,50.000000,393.000000,1.172000e+03
75%,4.046210e+06,0.000000,0.990000,7.000000,11.000000,55.000000,64.000000,1658.000000,1.212385e+03
max,6.114330e+06,16.000000,2.010000,15.000000,100.000000,100.000000,230.000000,27000.000000,4.075401e+06


In [7]:
# Store files in Parquet format for use in later scripts

# Frequency
fmv_freq_df.to_parquet("fmv_freq_df.parquet")

# Severity
fmv_sev_df.to_parquet("fmv_sev_df.parquet")

# Combined Frequency and Severity
fmv_df.to_parquet("fmv_df.parquet")